# Applied Unsupervised Learning Techniques

## Imports

In [1]:
import numpy as np
import altair as alt
import pandas as pd
import umap
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering

from birds.source_data import nabbp, avonet

/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [2]:
RANDOM_STATE = 42

In [3]:
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

## Data Preprocessing

### Load NABBP Species Data

In [4]:
def load_nabbp_species() -> pd.DataFrame:
    lookup = nabbp.LookupTables()
    df = lookup.species
    return df[df['ENDANGERED'] == 'Y']
    

nabbp_species_df = load_nabbp_species()
nabbp_species_df.head()

,SPECIES_ID,SPECIES_NAME,ALPHA_CODE,TAXONOMIC_ORDER,SCI_NAME,RECOMENDSIZE,ALLOWABLESIZE,ENDANGERED,RAPTOR,GAMEBIRD
24,230,Marbled Murrelet,MAMU,391.0,Brachyramphus marmoratus,"3, 3B","3, 3B",Y,NaN,NaN
79,720,Roseate Tern,ROST,455.0,Sterna dougallii,"2, 2A","2, 2A, 3",Y,NaN,NaN
86,743,California Least Tern,CLTE,449.0,Sternula antillarum browni,"1A, 1B","1A, 1B, 1P",Y,NaN,NaN
100,820,Short-tailed Albatross,STAL,131.0,Phoebastria albatrus,8,8,Y,NaN,NaN
115,932,Newell's Shearwater,NESH,172.0,Puffinus newelli,"4, 4A","4, 4A",Y,NaN,NaN


### Load Avonet Bird Tree Data

In [5]:
def load_avonet_data() -> pd.DataFrame:
    df = avonet.DataTables().bird_tree
    return df

avonet_df = load_avonet_data()
avonet_df.head()

,Species3,Family3,Order3,Total.individuals,Female,Male,Unknown,Complete.measures,Beak.Length_Culmen,Beak.Length_Nares,...,Migration,Trophic.Level,Trophic.Niche,Primary.Lifestyle,Min.Latitude,Max.Latitude,Centroid.Latitude,Centroid.Longitude,Range.Size,Species.Status
0,Accipiter albogularis,Accipitridae,Accipitriformes,5,2,0,3,4,27.7,17.8,...,2.0,Carnivore,Vertivore,Insessorial,-11.73,-4.02,-8.15,158.493765,37461.21,Extant
1,Accipiter badius,Accipitridae,Accipitriformes,10,4,6,0,8,20.6,12.1,...,3.0,Carnivore,Vertivore,Insessorial,-29.47,46.39,8.23,44.982464,22374973.00,Extant
2,Accipiter bicolor,Accipitridae,Accipitriformes,6,2,2,2,4,26.5,14.8,...,2.0,Carnivore,Vertivore,Generalist,NaN,NaN,NaN,NaN,NaN,Extant
3,Accipiter brachyurus,Accipitridae,Accipitriformes,4,4,0,0,3,22.5,14.0,...,2.0,Carnivore,Vertivore,Insessorial,-6.31,-4.08,-5.45,150.681314,35580.71,Extant
4,Accipiter brevipes,Accipitridae,Accipitriformes,8,4,4,0,4,21.1,12.1,...,3.0,Carnivore,Vertivore,Generalist,31.19,55.86,45.24,45.327340,2936751.80,Extant


### Merge Avonet and NABBP Datasets

#### Compute dataset overlap using the scientific name column

In [6]:
def compute_overlap(avonet_data: pd.DataFrame, nabbp_data: pd.DataFrame):
    bt= set(avonet_data['Species3'].unique())
    d = set(nabbp_data['SCI_NAME'].unique())

    overlap  = bt.intersection(d)
    return len(overlap)
    

compute_overlap(avonet_df, nabbp_species_df)

43

### Transform and Clean Merged Dataset

In [7]:
def merge_datasets(avonet_data: pd.DataFrame, nabbp_data: pd.DataFrame) -> pd.DataFrame:
    merged_df = pd.merge(avonet_data, nabbp_data, left_on='Species3', right_on='SCI_NAME', how='left')
    return merged_df


merged_df = merge_datasets(avonet_df, nabbp_species_df)
merged_df.head()

,Species3,Family3,Order3,Total.individuals,Female,Male,Unknown,Complete.measures,Beak.Length_Culmen,Beak.Length_Nares,...,SPECIES_ID,SPECIES_NAME,ALPHA_CODE,TAXONOMIC_ORDER,SCI_NAME,RECOMENDSIZE,ALLOWABLESIZE,ENDANGERED,RAPTOR,GAMEBIRD
0,Accipiter albogularis,Accipitridae,Accipitriformes,5,2,0,3,4,27.7,17.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Accipiter badius,Accipitridae,Accipitriformes,10,4,6,0,8,20.6,12.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Accipiter bicolor,Accipitridae,Accipitriformes,6,2,2,2,4,26.5,14.8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Accipiter brachyurus,Accipitridae,Accipitriformes,4,4,0,0,3,22.5,14.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Accipiter brevipes,Accipitridae,Accipitriformes,8,4,4,0,4,21.1,12.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
def transform_merge_dataset(merged_data: pd.DataFrame) -> pd.DataFrame:
    avonet_feature_columns = [
        'Beak.Length_Culmen',
        'Beak.Length_Nares',
        'Beak.Width',
        'Beak.Depth',
        'Tarsus.Length',
        'Wing.Length',
        'Kipps.Distance',
        'Hand-Wing.Index',
        'Tail.Length',
        'Mass',
    ]
    id_columns = [
        'SPECIES_ID',
        'Species3',
        'Family3',
        'Order3',
        'ENDANGERED',
    ]
    columns_to_keep = id_columns + avonet_feature_columns
    cleaned = (
        merged_data[columns_to_keep]
            .rename(columns={
                "ENDANGERED": "is_endangered",
                "SPECIES_ID": "species_id",
                "Species3": "species_name",
                "Family3": "family_name",
                "Order3": "order_name",
                "Beak.Length_Culmen": "beak_length_culmen",
                "Beak.Length_Nares": "beak_length_nares",
                "Beak.Width": "beak_width",
                "Beak.Depth": "beak_depth",
                "Tarsus.Length": "tarsus_length",
                "Wing.Length": "wing_length",
                "Kipps.Distance": "kipps_distance",
                "Hand-Wing.Index": "hand_wing_index",
                "Tail.Length": "tail_length",
                "Mass": "mass",
            })
            .fillna({
                "is_endangered": 'N',
            })
    )
    return cleaned

transformed_df = transform_merge_dataset(merged_df)
transformed_df.head()

,species_id,species_name,family_name,order_name,is_endangered,beak_length_culmen,beak_length_nares,beak_width,beak_depth,tarsus_length,wing_length,kipps_distance,hand_wing_index,tail_length,mass
0,NaN,Accipiter albogularis,Accipitridae,Accipitriformes,N,27.7,17.8,10.6,14.7,62.0,235.2,81.8,33.9,169.0,248.75
1,NaN,Accipiter badius,Accipitridae,Accipitriformes,N,20.6,12.1,8.8,11.6,43.0,186.7,62.5,32.9,140.6,131.15
2,NaN,Accipiter bicolor,Accipitridae,Accipitriformes,N,26.5,14.8,9.2,13.5,57.5,231.8,46.4,19.8,188.4,287.54
3,NaN,Accipiter brachyurus,Accipitridae,Accipitriformes,N,22.5,14.0,8.9,11.9,61.2,202.2,64.1,31.7,140.8,142.00
4,NaN,Accipiter brevipes,Accipitridae,Accipitriformes,N,21.1,12.1,8.7,11.1,46.4,217.6,87.8,40.2,153.5,186.48


In [9]:
AVONET_FEATURE_COLUMNS = [
    'beak_length_culmen',
    'beak_length_nares',
    'beak_width',
    'beak_depth',
    'tarsus_length',
    'wing_length',
    'kipps_distance',
    'hand_wing_index',
    'tail_length',
    'mass',
]

In [10]:
# Create a density plot for each numeric feature
charts = []
for col in AVONET_FEATURE_COLUMNS:
    chart = (
        alt.Chart(transformed_df)
            .transform_density(col, as_=[col, 'density'])
            .mark_area(opacity=0.6)
            .encode(x=alt.X(col, title=col), y='density:Q')
            .properties(width=150, height=100)
    )
    charts.append(chart)
alt.concat(*charts, columns=3, title="Density Plots of Avonet Features")

alt.ConcatChart(...)

### Feature Scaling

In [11]:
def scale_feature_columns(data: pd.DataFrame, feature_columns: list[str]):
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(data[feature_columns])

    data_scaled = data.copy()
    data_scaled[feature_columns] = scaled_features
    
    return data_scaled


scaled_df = scale_feature_columns(transformed_df, AVONET_FEATURE_COLUMNS)
scaled_df.head()

,species_id,species_name,family_name,order_name,is_endangered,beak_length_culmen,beak_length_nares,beak_width,beak_depth,tarsus_length,wing_length,kipps_distance,hand_wing_index,tail_length,mass
0,NaN,Accipiter albogularis,Accipitridae,Accipitriformes,N,0.047938,0.031854,0.771048,0.857926,1.315299,1.137242,0.943149,0.544636,1.310589,-0.012478
1,NaN,Accipiter badius,Accipitridae,Accipitriformes,N,-0.235911,-0.244608,0.424810,0.455207,0.557427,0.630570,0.528560,0.478252,0.855076,-0.085236
2,NaN,Accipiter bicolor,Accipitridae,Accipitriformes,N,-0.000036,-0.113652,0.501752,0.702035,1.135803,1.101723,0.182712,-0.391379,1.621749,0.011521
3,NaN,Accipiter brachyurus,Accipitridae,Accipitriformes,N,-0.159951,-0.152454,0.444045,0.494180,1.283389,0.792496,0.562930,0.398591,0.858284,-0.078523
4,NaN,Accipiter brevipes,Accipitridae,Accipitriformes,N,-0.215921,-0.244608,0.405574,0.390252,0.693046,0.953378,1.072036,0.962855,1.061982,-0.051004


### UMAP Projection for Visualizations

In [12]:
def compute_umap_projection(data: pd.DataFrame, feature_columns: list[str]) -> pd.DataFrame:
    reducer = umap.UMAP(
        random_state=RANDOM_STATE,
        min_dist=0.1,
        n_neighbors=15,
        metric='euclidean'
    )
    embedding = reducer.fit_transform(data[feature_columns])

    data_umap = data.copy()
    data_umap['umap_component_1'] = embedding[:, 0]
    data_umap['umap_component_2'] = embedding[:, 1]
    return data_umap


In [ ]:
umap_df = compute_umap_projection(scaled_df, AVONET_FEATURE_COLUMNS)
umap_df.head()

/venv/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## Visualizations

In [ ]:
def plot_umap(data: pd.DataFrame, color: alt.Color, title: str) -> alt.Chart:
    chart = (
    alt.Chart(data)
        .mark_circle(size=30)
        .encode(
            x=alt.X('umap_component_1'),
            y=alt.Y('umap_component_2'),
            color=color,
            tooltip=['family_name', 'order_name', 'is_endangered']
        )
        .properties(
            title=title,
            width=600,
            height=500
        )
    )
    return chart

In [ ]:
bird_order_plot = plot_umap(umap_df, alt.Color('order_name', legend=None), 'Bird Orders')
bird_family_plot = plot_umap(umap_df, alt.Color('family_name', legend=None), 'Bird Families')

plot = (
    (bird_order_plot | bird_family_plot)
        .interactive()
        .resolve_scale(x='shared', y='shared')
)
plot

In [ ]:
#plot_umap(umap_df, alt.Color('is_endangered'), 'Endangered Status').interactive()

## K-Means Clustering
Evaluation Metrics:
1. Cluster cardinality is the number of examples per cluster. Plot the cluster cardinality for all clusters and investigate clusters that are major outliers.
2. Cluster magnitude is the sum of distances from all examples in a cluster to the cluster's centroid. Plot cluster magnitude for all clusters and investigate outliers.
3. Within-Cluster Sum of Squares (WCSS), also known as inertia in scikit-learn, measures how compact your clusters are.
   - Low WCSS -> data points are close to their centroids, clusters are tight and cohesive.
   - High WCSS -> clusters are spread out, possibly overlapping.

In [ ]:
def plot_elbow(data: pd.DataFrame, n_clusters: int, feature_columns: list[str]) -> alt.Chart:
    """Plots the elbow curve for KMeans clustering to determine the optimal number of clusters.
    """
    input_data = data[feature_columns]
    inertia = []
    for n in range(1, n_clusters + 1):
        kmeans = KMeans(n_clusters=n, random_state=RANDOM_STATE)
        kmeans.fit(input_data)
        inertia.append(kmeans.inertia_) 

    df = pd.DataFrame({
        'n_clusters': list(range(1, n_clusters + 1)),
        'inertia': inertia
    })
    chart = (
        alt.Chart(df)
            .mark_line(point=True)
            .encode(
                x=alt.X('n_clusters', title='Number of Clusters'),
                y=alt.Y('inertia', title='Inertia')
            )
            .properties(
                title='Elbow Method for Optimal Number of Clusters (K)',
                width=600,
                height=400
            )
    )
    return chart

plot_elbow(umap_df, 60, AVONET_FEATURE_COLUMNS)


In [ ]:
K = 40

In [ ]:
def compute_kmeans_clusters(data: pd.DataFrame, n_clusters: int, feature_columns: list[str]) -> pd.DataFrame:
    """Computes KMeans clustering and returns the data with cluster labels and cluster statistics.
    """
    input_data = data[feature_columns]
    kmeans = KMeans(
        n_clusters=n_clusters, 
        random_state=RANDOM_STATE
    )
    
    data_with_clusters = data.copy()
    data_with_clusters['cluster'] = kmeans.fit_predict(input_data)

    centroids = kmeans.cluster_centers_
    cardinality = pd.Series(kmeans.labels_).value_counts().sort_index()
    
    # Compute average distance and WCSS per cluster
    magnitudes = []
    wcss = []
    for i in range(len(centroids)):
        cluster_points = input_data[kmeans.labels_ == i]
        distances = np.linalg.norm(cluster_points - centroids[i], axis=1)

        magnitudes.append(distances.mean())
        wcss.append(np.sum(distances ** 2))
        
    cluster_stats = pd.DataFrame({
        'cluster': range(len(centroids)),
        'Cardinality': cardinality.values,
        'Magnitude (avg distance)': magnitudes,
        'WCSS (inertia)': wcss
    })

    return data_with_clusters, cluster_stats


clustered_df, cluster_stats = compute_kmeans_clusters(data=umap_df, n_clusters=K, feature_columns=AVONET_FEATURE_COLUMNS)
cluster_stats.head(100)

In [ ]:
def plot_cluster_stats(data: pd.DataFrame):
    """Plots cluster statistics: Cardinality, Magnitude, and WCSS."""
    color = alt.Color('cluster:N', legend=alt.Legend(title="Cluster"), scale=alt.Scale(scheme='category20'))

    base = (
        alt.Chart(data)
            .encode(
                x=alt.X('cluster:N', title='Cluster', axis=alt.Axis(labelAngle=0))  # keep labels horizontal
            )
    )

    cardinality_chart = (
        base.mark_bar()
        .encode(
            y=alt.Y('Cardinality:Q', title='Cardinality'),
            color=color
        )
        .properties(height=120)
    )

    magnitude_chart = (
        base.mark_bar()
        .encode(
            y=alt.Y('Magnitude (avg distance):Q', title='Magnitude (Avg Distance)'),
            color=color
        )
        .properties(height=120)
    )

    wcss_chart = (
        base.mark_bar()
        .encode(
            y=alt.Y('WCSS (inertia):Q', title='WCSS (Inertia)'),
            color=color
        )
        .properties(height=120)
    )

    final_chart = alt.vconcat(cardinality_chart, magnitude_chart, wcss_chart)
    return final_chart, color

In [ ]:
cluster_stats_plot, color = plot_cluster_stats(cluster_stats)
chart = (
    (plot_umap(clustered_df, color, 'Clustered') | cluster_stats_plot)
        .interactive()
        .resolve_scale(x='shared', y='shared', color='shared')
)

chart

In [ ]:
def categorize_clusters(df):
    summary = (
        df.groupby('cluster')['is_endangered']
        .agg(
            total='count',
            endangered_count=lambda x: (x == 'Y').sum(),
            non_endangered_count=lambda x: (x == 'N').sum()
        )
    )
    def classify(row):
        """Classify clusters based on endangered status composition."""
        if row['endangered_count'] == 0:
            return 'No Endangered'
        elif row['non_endangered_count'] == 0:
            return 'Only Endangered'
        else:
            return 'Mixed'

    summary['cluster_type'] = summary.apply(classify, axis=1)
    return summary.reset_index()

cluster_summary = categorize_clusters(clustered_df)
print(cluster_summary)

In [ ]:
clustered_df['cluster_type'] = clustered_df['cluster'].map(cluster_summary.set_index('cluster')['cluster_type'])
clustered_df.head()

In [ ]:
plot_umap(clustered_df, alt.Color('cluster_type', legend=alt.Legend(title="Cluster Type")), 'Cluster Types').interactive()

## Label Preparation

In [ ]:
clustered_df['is_endangered_adjusted'] = np.where(clustered_df['is_endangered'] == 'Y', 'Y', np.where(clustered_df['cluster_type'] == 'No Endangered', 'N', 'N/A'))
clustered_df['is_endangered_adjusted'].value_counts()

In [ ]:
import warnings

warnings.filterwarnings('ignore')

In [ ]:
from sklearn.semi_supervised import LabelPropagation
from sklearn.model_selection import GridSearchCV


def compute_label_propagation(data: pd.DataFrame, feature_columns: list[str]) -> pd.DataFrame:
    """
    """
    input_data = data[feature_columns].to_numpy()
    label_map = {'N': 0, 'Y': 1}
    inv_label_map = {0: 'N', 1: 'Y'}

    y = np.full(len(data), -1, dtype=int)  # start as unlabeled
    mask_labeled = data['is_endangered_adjusted'].isin(['Y', 'N'])
    y[mask_labeled] = data.loc[mask_labeled, 'is_endangered_adjusted'].map(label_map).to_numpy()

    # param_grid = {
    #     'gamma': [25, 50, 150, 200],
    #     'max_iter': [100, 200, 300],
    #     'kernel': ['rbf']
    # }

    
    # Best parameters: {'gamma': 200, 'kernel': 'rbf', 'max_iter': 100}
    param_grid = {
        'gamma': [200],
        'max_iter': [100],
        'kernel': ['rbf']
    }
    grid_search = GridSearchCV(
        estimator=LabelPropagation(),
        param_grid=param_grid,
        cv=5,
        scoring='accuracy'
    )
    
    grid_search.fit(input_data, y)

    print(f"Best score: {grid_search.best_score_:.3f}")
    print(f"Best parameters: {grid_search.best_params_}")

    best_model = grid_search.best_estimator_
    
    pred_all = best_model.transduction_                 # labels for all rows (0/1), respecting fixed seeds
    proba_all = best_model.label_distributions_         # class probabilities for all rows

    
    output_data = data.copy()
    pred_str = pd.Series(pred_all).map(inv_label_map) # Map back to Y/N strings and only fill where original was N/A

    output_data['is_endangered_propagated'] = output_data['is_endangered_adjusted']  # keep your originals
    na_mask = output_data['is_endangered_adjusted'].eq('N/A')
    output_data.loc[na_mask, 'is_endangered_propagated'] = pred_str[na_mask].values

    output_data['propagated_prob_Y'] = proba_all[:, 1]

    return output_data



propagated_df = compute_label_propagation(clustered_df, AVONET_FEATURE_COLUMNS)
propagated_df.head()

In [ ]:
high_conf = propagated_df[propagated_df['propagated_prob_Y'].between(0.9, 1)]
low_conf  = propagated_df[propagated_df['propagated_prob_Y'].between(0.0, 0.1)]

In [ ]:
low_conf.shape, high_conf.shape

In [ ]:
high_conf.head(100)

In [ ]:
propagated_df.sort_values('propagated_prob_Y')[['species_name', 'is_endangered_adjusted', 'is_endangered_propagated', 'propagated_prob_Y']].head(100)

In [ ]:
propagated_df['is_endangered_propagated'].value_counts()

In [ ]:
plot_umap(propagated_df, alt.Color('is_endangered_propagated', legend=alt.Legend(title="Propagated Endangered Status")), '').interactive()